[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/01_baseline_eval.ipynb)

# Notebook 1 — Baseline Evaluation

Measure LFM2-350M accuracy on StepGame **before** fine-tuning.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/spatialft.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/spatialft/spatialft.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.colab_utils import prepare_notebook, publish_artifacts

REPO, PATHS = prepare_notebook(REPO)
print(f'Repo ready at {REPO}')


In [ ]:
# Optional in Colab:
# !pip install -q -r ../requirements.txt
print(f'Using local repo storage: {PATHS["repo_root"]}')


In [ ]:
import json
import shutil
from pathlib import Path

import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.dataset import format_prompt, load_stepgame
from src.eval import evaluate, save_results


In [ ]:
MODEL_ID = 'LiquidAI/LFM2-350M'
EVAL_PATH = PATHS['data_root'] / 'eval' / 'stepgame_eval.json'
OUT_PATH = PATHS['results_root'] / 'baseline' / 'predictions.json'
SCORES_PATH = PATHS['results_root'] / 'baseline' / 'scores.json'
MAX_NEW_TOKENS = 64  # Direct-answer prompting keeps generations short for the 350M model
BATCH_SIZE = 8


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

In [ ]:
examples = load_stepgame(EVAL_PATH)
print(f'Loaded {len(examples)} eval examples')
print('Sample:', examples[0])

In [ ]:
predictions = []

for i in tqdm(range(0, len(examples), BATCH_SIZE)):
    batch = examples[i : i + BATCH_SIZE]
    prompts = [format_prompt(ex['story'], ex['question']) for ex in batch]

    inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs['input_ids'].shape[1]
    for ex, output in zip(batch, outputs):
        generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        predictions.append({
            'story': ex['story'],
            'question': ex['question'],
            'answer': ex['answer'],
            'prediction': generated,
            'k': ex.get('k'),
        })

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w') as f:
    json.dump(predictions, f, indent=2)


In [ ]:
results = evaluate(predictions)
save_results(results, SCORES_PATH)

print(f"Overall accuracy: {results['accuracy']:.3f}")
for k, v in results.items():
    if k.startswith('accuracy_k'):
        print(f"  {k}: {v:.3f}")


In [ ]:
publish_artifacts(
    [
        'results/baseline/predictions.json',
        'results/baseline/scores.json',
    ],
    'Add baseline predictions and scores [notebook 01]',
    repo_dir=REPO,
)
